# MLP - variaveis fisicas com gold

Este notebook usa exclusivamente `data/gold/inmet_pe_daily.csv` para treinar um MLP multi-saida que prediz irradiacao solar diaria e velocidade media do vento. A geracao em kWh e calculada depois da predicao fisica.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import mlflow.sklearn
import pandas as pd
from threadpoolctl import threadpool_limits

from src.modeling.gold_energy import (
    ENERGY_RESULT_COLUMNS,
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    assert_no_forbidden_features,
    build_prediction_results_table,
    calculate_energy_outputs_from_physical,
    clip_physical_predictions,
    configure_mlflow_tracking,
    describe_search_space,
    evaluate_predictions,
    fit_final_model,
    load_gold_daily,
    make_future_feature_frame,
    make_temporal_cv_splits,
    prepare_energy_modeling_table,
    split_feature_target_metadata,
    temporal_train_test_split,
    to_jsonable,
    train_random_search,
    mlp_base_estimator,
    mlp_refinement_space,
    mlp_search_space,
)
from src.modeling.historical_features import (
    ClimatologyBaselineRegressor,
    HISTORICAL_FEATURE_COLUMNS,
    compare_metric_tables,
    fit_historical_feature_reference,
    prepare_historical_train_test_frames,
    save_historical_reference,
    split_feature_target_metadata_with_columns,
    transform_with_historical_features,
)
from src.modeling.training_config import (
    BLAS_THREADS,
    CPU_WORKERS,
    ENERGY_CONFIG,
    FUTURE_DATE,
    FUTURE_STATION_CODE,
    HISTORY_MIN_OBSERVATIONS_DAY,
    HISTORY_MIN_OBSERVATIONS_MONTH,
    MLFLOW_EXPERIMENT_NAME,
    PRODUCTION_REFIT_WITH_FULL_GOLD,
    TEST_YEAR_FRACTION,
    USE_HISTORICAL_FEATURES,
)

RUN_ARTIFACTS_DIR = configure_mlflow_tracking(PROJECT_ROOT, MLFLOW_EXPERIMENT_NAME)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretorio de artefatos de modelagem: {RUN_ARTIFACTS_DIR}")

C:\Users\Admin\Desktop\Projetos\Projeto-ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raiz do projeto: C:\Users\Admin\Desktop\Projetos\Projeto-ML
Diretorio de artefatos de modelagem: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558


In [2]:
# Configuracoes compartilhadas ficam em src/modeling/training_config.py.
# A semente aleatoria fica local no notebook, para permitir reprodutibilidade por modelo.
MODEL_NAME = "mlp"
RANDOM_STATE = 42
N_ITER_RANDOM = 40
N_ITER_REFINEMENT = 15

assert CPU_WORKERS > 0, "CPU_WORKERS deve ser positivo."
assert BLAS_THREADS > 0, "BLAS_THREADS deve ser positivo."
print(f"CPU_WORKERS={CPU_WORKERS}; BLAS_THREADS={BLAS_THREADS}; threads planejadas={CPU_WORKERS * BLAS_THREADS}")
print(f"USE_HISTORICAL_FEATURES={USE_HISTORICAL_FEATURES}")

CPU_WORKERS=8; BLAS_THREADS=2; threads planejadas=16
USE_HISTORICAL_FEATURES=True


In [3]:
daily_gold = load_gold_daily(PROJECT_ROOT)
modeling_table = prepare_energy_modeling_table(daily_gold, ENERGY_CONFIG)
X_base, y_base, metadata_base = split_feature_target_metadata(modeling_table)

(
    X_train_base,
    X_test_base,
    y_train_base,
    y_test_base,
    train_metadata_base,
    test_metadata_base,
    train_years,
    test_years,
) = temporal_train_test_split(X_base, y_base, metadata_base, test_year_fraction=TEST_YEAR_FRACTION)

baseline_model = ClimatologyBaselineRegressor(
    min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
    min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
)
X_train_base_for_baseline = X_train_base.copy()
X_train_base_for_baseline["year"] = train_metadata_base["year"].to_numpy()
baseline_model.fit(X_train_base_for_baseline, y_train_base)

if USE_HISTORICAL_FEATURES:
    train_frame, test_frame, train_history_reference = prepare_historical_train_test_frames(
        modeling_table,
        train_years,
        test_years,
        min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
        min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
    )
    model_feature_columns = FEATURE_COLUMNS + HISTORICAL_FEATURE_COLUMNS
    X_train, y_train, train_metadata = split_feature_target_metadata_with_columns(train_frame, model_feature_columns)
    X_test, y_test, test_metadata = split_feature_target_metadata_with_columns(test_frame, model_feature_columns)
else:
    train_history_reference = None
    model_feature_columns = FEATURE_COLUMNS
    X_train = X_train_base
    X_test = X_test_base
    y_train = y_train_base
    y_test = y_test_base
    train_metadata = train_metadata_base
    test_metadata = test_metadata_base

baseline_predictions = baseline_model.predict(X_test[FEATURE_COLUMNS])
baseline_predictions = clip_physical_predictions(baseline_predictions)
baseline_metrics_table, baseline_metrics = evaluate_predictions(y_test, baseline_predictions)
baseline_results, baseline_energy_actual, baseline_energy_pred, _ = build_prediction_results_table(
    test_metadata,
    y_test,
    baseline_predictions,
    ENERGY_CONFIG,
)
baseline_energy_metrics_table, baseline_energy_metrics = evaluate_predictions(
    baseline_energy_actual[ENERGY_RESULT_COLUMNS],
    baseline_energy_pred[ENERGY_RESULT_COLUMNS],
    target_columns=ENERGY_RESULT_COLUMNS,
)

assert_no_forbidden_features(model_feature_columns)
cv_splits = make_temporal_cv_splits(train_metadata)
effective_train_years = sorted(train_metadata["year"].dropna().astype(int).unique().tolist())

print(f"Linhas gold usadas: {len(modeling_table):,}")
print(f"Linhas de treino do modelo: {len(X_train):,}")
print(f"Linhas de teste final: {len(X_test):,}")
print(f"Variaveis de entrada: {model_feature_columns}")
print(f"Alvos fisicos do ML: {TARGET_COLUMNS}")
print(f"Saidas energeticas calculadas: {ENERGY_RESULT_COLUMNS}")
print(f"Anos treino/validacao originais: {train_years}")
print(f"Anos treino efetivos do modelo: {effective_train_years}")
print(f"Anos teste final: {test_years}")
print(f"Folds temporais no treino: {len(cv_splits)}")
print("Metricas fisicas da baseline no mesmo teste temporal:")
display(baseline_metrics_table)
print("Metricas energeticas calculadas da baseline:")
display(baseline_energy_metrics_table)

Linhas gold usadas: 58,022
Linhas de treino do modelo: 51,430
Linhas de teste final: 6,278
Variaveis de entrada: ['station_code', 'latitude', 'longitude', 'altitude', 'month', 'day_of_year', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos', 'hist_solar_irradiation_station_doy_mean', 'hist_solar_irradiation_station_doy_median', 'hist_solar_irradiation_station_doy_std', 'hist_solar_irradiation_station_doy_count', 'hist_solar_irradiation_station_month_mean', 'hist_solar_irradiation_station_month_median', 'hist_solar_irradiation_station_month_std', 'hist_solar_irradiation_station_month_count', 'hist_solar_irradiation_station_mean', 'hist_solar_irradiation_station_median', 'hist_solar_irradiation_station_std', 'hist_solar_irradiation_station_count', 'hist_solar_irradiation_global_doy_mean', 'hist_solar_irradiation_global_doy_median', 'hist_solar_irradiation_global_doy_std', 'hist_solar_irradiation_global_doy_count', 'hist_solar_irradiation_global_month_mean', 'hist_solar_irrad

,target,mae,rmse,nrmse,r2,bias,medae,smape
0,solar_daily_kwh_m2_day,1.340608,1.696688,0.164762,-0.113885,0.150524,1.127885,27.182303
1,wind_daily_mean_ms,0.873305,1.174111,0.176411,-0.234772,0.411390,0.650388,38.179938


Metricas energeticas calculadas da baseline:


,target,mae,rmse,nrmse,r2,bias,medae,smape
0,solar_generation_kwh_day,233.547481,295.580117,0.164762,-0.113885,26.222758,196.488936,27.182303
1,wind_generation_kwh_day,4297.198228,6415.694604,0.095312,-0.120071,840.685409,3015.222139,81.562432
2,hybrid_generation_kwh_day,4333.153921,6450.438357,0.094740,-0.126437,866.908166,3039.771925,65.784502


In [4]:
mlp_estimator = mlp_base_estimator(random_state=RANDOM_STATE, feature_columns=model_feature_columns)
mlp_space = mlp_search_space()
print("Espaco amplo MLP:", describe_search_space(mlp_space))

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="mlp_physical_gold_hist"):
    mlflow_run_id = mlflow.active_run().info.run_id
    mlflow.log_param("model_family", "MLPRegressor")
    mlflow.log_param("cpu_workers", CPU_WORKERS)
    mlflow.log_param("blas_threads", BLAS_THREADS)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("n_iter_random", N_ITER_RANDOM)
    mlflow.log_param("n_iter_refinement", N_ITER_REFINEMENT)
    mlflow.log_param("use_historical_features", USE_HISTORICAL_FEATURES)
    mlflow.log_param("history_min_observations_day", HISTORY_MIN_OBSERVATIONS_DAY)
    mlflow.log_param("history_min_observations_month", HISTORY_MIN_OBSERVATIONS_MONTH)
    mlflow.log_param("train_years", ",".join(map(str, train_years)))
    mlflow.log_param("effective_train_years", ",".join(map(str, effective_train_years)))
    mlflow.log_param("test_years", ",".join(map(str, test_years)))
    mlflow.log_param("targets", ",".join(TARGET_COLUMNS))
    mlflow.log_param("energy_outputs", ",".join(ENERGY_RESULT_COLUMNS))
    mlflow.log_param("features", ",".join(model_feature_columns))
    mlflow.log_dict(to_jsonable(ENERGY_CONFIG), "energy_config.json")
    for metric_name, metric_value in baseline_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"baseline_physical_{metric_name}", float(metric_value))
    for metric_name, metric_value in baseline_energy_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"baseline_energy_{metric_name}", float(metric_value))

    with threadpool_limits(limits=BLAS_THREADS):
        broad_search, broad_seconds = train_random_search(
            mlp_estimator,
            mlp_space,
            X_train,
            y_train,
            cv_splits,
            n_iter=N_ITER_RANDOM,
            n_jobs=CPU_WORKERS,
            random_state=RANDOM_STATE,
        )

    mlp_ref_space = mlp_refinement_space(broad_search.best_params_)
    mlflow.log_metric("broad_search_seconds", broad_seconds)
    mlflow.log_metric("broad_best_balanced_negative_nrmse", broad_search.best_score_)
    mlflow.log_dict(to_jsonable(broad_search.best_params_), "best_params_broad.json")
    mlflow.log_dict(to_jsonable(mlp_ref_space), "search_space_refinement.json")

    with threadpool_limits(limits=BLAS_THREADS):
        refine_search, refine_seconds = train_random_search(
            mlp_estimator,
            mlp_ref_space,
            X_train,
            y_train,
            cv_splits,
            n_iter=N_ITER_REFINEMENT,
            n_jobs=CPU_WORKERS,
            random_state=RANDOM_STATE + 1,
        )

    with threadpool_limits(limits=BLAS_THREADS):
        final_model, final_fit_seconds = fit_final_model(
            mlp_estimator,
            refine_search.best_params_,
            X_train,
            y_train,
        )

    mlflow.log_metric("refine_best_balanced_negative_nrmse", refine_search.best_score_)
    mlflow.log_dict(to_jsonable(refine_search.best_params_), "best_params_refinement.json")

    test_predictions = clip_physical_predictions(final_model.predict(X_test))
    metrics_table, final_metrics = evaluate_predictions(y_test, test_predictions)
    test_results, energy_actual, energy_pred, baseline_energy_for_model = build_prediction_results_table(
        test_metadata,
        y_test,
        test_predictions,
        ENERGY_CONFIG,
        baseline_predictions=baseline_predictions,
    )
    energy_metrics_table, energy_metrics = evaluate_predictions(
        energy_actual[ENERGY_RESULT_COLUMNS],
        energy_pred[ENERGY_RESULT_COLUMNS],
        target_columns=ENERGY_RESULT_COLUMNS,
    )
    comparison_table = compare_metric_tables(metrics_table, baseline_metrics_table, MODEL_NAME)
    energy_comparison_table = compare_metric_tables(energy_metrics_table, baseline_energy_metrics_table, MODEL_NAME)
    for metric_name, metric_value in final_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"physical_{metric_name}", float(metric_value))
    for metric_name, metric_value in energy_metrics.items():
        if pd.notna(metric_value):
            mlflow.log_metric(f"energy_{metric_name}", float(metric_value))
    mlflow.log_metric("refine_search_seconds", refine_seconds)
    mlflow.log_metric("final_fit_seconds", final_fit_seconds)
    mlflow.sklearn.log_model(final_model, artifact_path="model")

search_summary = pd.DataFrame(
    [
        {"fase": "busca_ampla", "balanced_negative_nrmse": broad_search.best_score_, "segundos": broad_seconds},
        {"fase": "refino", "balanced_negative_nrmse": refine_search.best_score_, "segundos": refine_seconds},
        {"fase": "treino_final", "balanced_negative_nrmse": None, "segundos": final_fit_seconds},
    ]
)
best_params_table = pd.Series(refine_search.best_params_, name="valor").rename_axis("hiperparametro").reset_index()

print("RESULTADOS FINAIS - MLP")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de treino/validacao: {train_years}")
print(f"Anos de treino efetivos: {effective_train_years}")
print(f"Anos de teste final: {test_years}")
print("Metricas fisicas no teste temporal final:")
display(metrics_table)
print("Metricas energeticas calculadas:")
display(energy_metrics_table)
print("Comparacao fisica contra baseline:")
display(comparison_table)
print("Comparacao energetica contra baseline:")
display(energy_comparison_table)
print("Resumo da busca de hiperparametros:")
display(search_summary)
print("Melhores hiperparametros do refino:")
display(best_params_table)

Espaco amplo MLP: {"regressor__model__activation": 2, "regressor__model__alpha": 80, "regressor__model__batch_size": 4, "regressor__model__early_stopping": 1, "regressor__model__hidden_layer_sizes": 7, "regressor__model__learning_rate": 2, "regressor__model__learning_rate_init": 80, "regressor__model__max_iter": 1, "regressor__model__n_iter_no_change": 2, "regressor__model__validation_fraction": 1}


Fitting 16 folds for each of 40 candidates, totalling 640 fits


Fitting 16 folds for each of 15 candidates, totalling 240 fits


2026/06/13 21:43:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/06/13 21:43:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

,target,metric,baseline,model,model_minus_baseline,improvement_pct
0,solar_generation_kwh_day,mae,233.547481,245.918300,12.370818,-5.296918
1,solar_generation_kwh_day,rmse,295.580117,317.361537,21.781420,-7.369041
2,solar_generation_kwh_day,nrmse,0.164762,0.176903,0.012141,-7.369041
3,solar_generation_kwh_day,medae,196.488936,196.535979,0.047043,-0.023942
4,solar_generation_kwh_day,smape,27.182303,30.255581,3.073278,-11.306173
5,solar_generation_kwh_day,r2,-0.113885,-0.284099,-0.170214,NaN
6,wind_generation_kwh_day,mae,4297.198228,3503.713493,-793.484734,18.465165
7,wind_generation_kwh_day,rmse,6415.694604,5574.322646,-841.371958,13.114277
8,wind_generation_kwh_day,nrmse,0.095312,0.082813,-0.012500,13.114277
9,wind_generation_kwh_day,medae,3015.222139,2226.737828,-788.484311,26.150123


Resumo da busca de hiperparametros:


,fase,balanced_negative_nrmse,segundos
0,busca_ampla,-0.184668,4494.964457
1,refino,-0.182995,4573.232035
2,treino_final,NaN,396.913720


Melhores hiperparametros do refino:


,hiperparametro,valor
0,regressor__model__validation_fraction,0.15
1,regressor__model__n_iter_no_change,40
2,regressor__model__max_iter,1000
3,regressor__model__learning_rate_init,0.001643
4,regressor__model__learning_rate,adaptive
5,regressor__model__hidden_layer_sizes,"(512, 256)"
6,regressor__model__early_stopping,True
7,regressor__model__batch_size,64
8,regressor__model__alpha,0.000143
9,regressor__model__activation,tanh


In [5]:
# Resumo explicito das metricas fisicas e energeticas no teste temporal final.
print("RESUMO DAS METRICAS - TESTE TEMPORAL FINAL")
print(f"Modelo: {MODEL_NAME}")
print(f"MLflow run_id: {mlflow_run_id}")
print(f"Anos de teste final: {test_years}")
print(f"physical_balanced_nrmse: {final_metrics['balanced_nrmse']:.6f}")
print(f"baseline_physical_balanced_nrmse: {baseline_metrics['balanced_nrmse']:.6f}")
print("")
print("Metricas fisicas previstas pelo ML:")
for _, row in metrics_table.iterrows():
    print(f"Alvo: {row['target']}")
    print(f"  MAE: {row['mae']:.6f}")
    print(f"  RMSE: {row['rmse']:.6f}")
    print(f"  NRMSE: {row['nrmse']:.6f}")
    print(f"  R2: {row['r2']:.6f}")
    print(f"  Bias medio: {row['bias']:.6f}")
    print(f"  MedAE: {row['medae']:.6f}")
    print(f"  sMAPE (%): {row['smape']:.6f}")
print("")
print("Metricas energeticas calculadas apos a predicao fisica:")
for _, row in energy_metrics_table.iterrows():
    print(f"Saida: {row['target']}")
    print(f"  MAE: {row['mae']:.6f}")
    print(f"  RMSE: {row['rmse']:.6f}")
    print(f"  NRMSE: {row['nrmse']:.6f}")
    print(f"  R2: {row['r2']:.6f}")
    print(f"  Bias medio: {row['bias']:.6f}")
    print(f"  MedAE: {row['medae']:.6f}")
    print(f"  sMAPE (%): {row['smape']:.6f}")

print("")
print("Metricas usadas na busca de hiperparametros:")
print(f"  broad_best_balanced_negative_nrmse: {broad_search.best_score_:.6f}")
print(f"  refine_best_balanced_negative_nrmse: {refine_search.best_score_:.6f}")
print(f"  broad_search_seconds: {broad_seconds:.2f}")
print(f"  refine_search_seconds: {refine_seconds:.2f}")
print(f"  final_fit_seconds: {final_fit_seconds:.2f}")

print("")
print("Comparacao fisica contra baseline:")
display(comparison_table)
print("Comparacao energetica contra baseline:")
display(energy_comparison_table)

RESUMO DAS METRICAS - TESTE TEMPORAL FINAL
Modelo: mlp
MLflow run_id: 016cfadf6aa340a59d950c6cf7073e9a
Anos de teste final: [2021, 2022, 2023, 2024, 2025]
physical_balanced_nrmse: 0.157521
baseline_physical_balanced_nrmse: 0.170586

Metricas fisicas previstas pelo ML:
Alvo: solar_daily_kwh_m2_day
  MAE: 1.411619
  RMSE: 1.821717
  NRMSE: 0.176903
  R2: -0.284099
  Bias medio: -0.138833
  MedAE: 1.128155
  sMAPE (%): 30.255581
Alvo: wind_daily_mean_ms
  MAE: 0.721518
  RMSE: 0.919387
  NRMSE: 0.138138
  R2: 0.242880
  Bias medio: 0.187781
  MedAE: 0.597660
  sMAPE (%): 34.899331

Metricas energeticas calculadas apos a predicao fisica:
Saida: solar_generation_kwh_day
  MAE: 245.918300
  RMSE: 317.361537
  NRMSE: 0.176903
  R2: -0.284099
  Bias medio: -24.186092
  MedAE: 196.535979
  sMAPE (%): 30.255581
Saida: wind_generation_kwh_day
  MAE: 3503.713493
  RMSE: 5574.322646
  NRMSE: 0.082813
  R2: 0.154444
  Bias medio: -284.116539
  MedAE: 2226.737828
  sMAPE (%): 78.013590
Saida: hybrid_

,target,metric,baseline,model,model_minus_baseline,improvement_pct
0,solar_daily_kwh_m2_day,mae,1.340608,1.411619,0.071011,-5.296918
1,solar_daily_kwh_m2_day,rmse,1.696688,1.821717,0.125030,-7.369041
2,solar_daily_kwh_m2_day,nrmse,0.164762,0.176903,0.012141,-7.369041
3,solar_daily_kwh_m2_day,medae,1.127885,1.128155,0.000270,-0.023942
4,solar_daily_kwh_m2_day,smape,27.182303,30.255581,3.073278,-11.306173
5,solar_daily_kwh_m2_day,r2,-0.113885,-0.284099,-0.170214,NaN
6,wind_daily_mean_ms,mae,0.873305,0.721518,-0.151786,17.380667
7,wind_daily_mean_ms,rmse,1.174111,0.919387,-0.254725,21.695104
8,wind_daily_mean_ms,nrmse,0.176411,0.138138,-0.038272,21.695104
9,wind_daily_mean_ms,medae,0.650388,0.597660,-0.052729,8.107259


Comparacao energetica contra baseline:


,target,metric,baseline,model,model_minus_baseline,improvement_pct
0,solar_generation_kwh_day,mae,233.547481,245.918300,12.370818,-5.296918
1,solar_generation_kwh_day,rmse,295.580117,317.361537,21.781420,-7.369041
2,solar_generation_kwh_day,nrmse,0.164762,0.176903,0.012141,-7.369041
3,solar_generation_kwh_day,medae,196.488936,196.535979,0.047043,-0.023942
4,solar_generation_kwh_day,smape,27.182303,30.255581,3.073278,-11.306173
5,solar_generation_kwh_day,r2,-0.113885,-0.284099,-0.170214,NaN
6,wind_generation_kwh_day,mae,4297.198228,3503.713493,-793.484734,18.465165
7,wind_generation_kwh_day,rmse,6415.694604,5574.322646,-841.371958,13.114277
8,wind_generation_kwh_day,nrmse,0.095312,0.082813,-0.012500,13.114277
9,wind_generation_kwh_day,medae,3015.222139,2226.737828,-788.484311,26.150123


In [6]:
test_results, energy_actual, energy_pred, baseline_energy_for_model = build_prediction_results_table(
    test_metadata,
    y_test,
    test_predictions,
    ENERGY_CONFIG,
    baseline_predictions=baseline_predictions,
)

results_dir = RUN_ARTIFACTS_DIR / "evaluation" / MODEL_NAME
results_dir.mkdir(parents=True, exist_ok=True)
run_timestamp = pd.Timestamp.now().strftime("%Y%m%d-%H%M%S")
metrics_output_path = results_dir / f"{MODEL_NAME}_physical_metrics_{run_timestamp}.csv"
energy_metrics_output_path = results_dir / f"{MODEL_NAME}_energy_metrics_{run_timestamp}.csv"
baseline_metrics_output_path = results_dir / f"{MODEL_NAME}_baseline_physical_metrics_{run_timestamp}.csv"
baseline_energy_metrics_output_path = results_dir / f"{MODEL_NAME}_baseline_energy_metrics_{run_timestamp}.csv"
comparison_output_path = results_dir / f"{MODEL_NAME}_baseline_physical_comparison_{run_timestamp}.csv"
energy_comparison_output_path = results_dir / f"{MODEL_NAME}_baseline_energy_comparison_{run_timestamp}.csv"
predictions_output_path = results_dir / f"{MODEL_NAME}_test_predictions_{run_timestamp}.csv"
predictions_sample_output_path = results_dir / f"{MODEL_NAME}_test_predictions_sample_{run_timestamp}.csv"

metrics_table.to_csv(metrics_output_path, index=False)
energy_metrics_table.to_csv(energy_metrics_output_path, index=False)
baseline_metrics_table.to_csv(baseline_metrics_output_path, index=False)
baseline_energy_metrics_table.to_csv(baseline_energy_metrics_output_path, index=False)
comparison_table.to_csv(comparison_output_path, index=False)
energy_comparison_table.to_csv(energy_comparison_output_path, index=False)
test_results.to_csv(predictions_output_path, index=False)
test_results.head(50).to_csv(predictions_sample_output_path, index=False)

with mlflow.start_run(run_id=mlflow_run_id):
    mlflow.log_artifact(str(metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(energy_metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(baseline_metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(baseline_energy_metrics_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(comparison_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(energy_comparison_output_path), artifact_path="evaluation")
    mlflow.log_artifact(str(predictions_sample_output_path), artifact_path="evaluation")
    if train_history_reference is not None:
        train_reference_dir = results_dir / "historical_reference_train"
        production_reference = (
            fit_historical_feature_reference(
                modeling_table,
                min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
                min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
            )
            if PRODUCTION_REFIT_WITH_FULL_GOLD
            else train_history_reference
        )
        production_reference_dir = results_dir / "historical_reference_production"
        save_historical_reference(train_history_reference, train_reference_dir)
        save_historical_reference(production_reference, production_reference_dir)
        mlflow.log_artifacts(str(train_reference_dir), artifact_path="historical_reference_train")
        mlflow.log_artifacts(str(production_reference_dir), artifact_path="historical_reference_production")

print("Arquivos de resultados salvos:")
print(f"- Metricas fisicas: {metrics_output_path}")
print(f"- Metricas energeticas: {energy_metrics_output_path}")
print(f"- Metricas fisicas da baseline: {baseline_metrics_output_path}")
print(f"- Metricas energeticas da baseline: {baseline_energy_metrics_output_path}")
print(f"- Comparacao fisica com baseline: {comparison_output_path}")
print(f"- Comparacao energetica com baseline: {energy_comparison_output_path}")
print(f"- Predicoes completas do teste: {predictions_output_path}")
print(f"- Amostra das predicoes: {predictions_sample_output_path}")
print("Primeiras predicoes do teste temporal:")
display(test_results.head(20))

Arquivos de resultados salvos:
- Metricas fisicas: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\mlp\mlp_physical_metrics_20260613-214358.csv
- Metricas energeticas: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\mlp\mlp_energy_metrics_20260613-214358.csv
- Metricas fisicas da baseline: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\mlp\mlp_baseline_physical_metrics_20260613-214358.csv
- Metricas energeticas da baseline: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\mlp\mlp_baseline_energy_metrics_20260613-214358.csv
- Comparacao fisica com baseline: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\mlp\mlp_baseline_physical_comparison_20260613-214358.csv
- Comparacao energetica com baseline: C:\Users\Admin\Desktop\Projetos\Projeto-ML\artifacts\modeling\20260613-134558\evaluation\

,station_code,station_name,city,state,latitude,longitude,altitude,date,year,solar_daily_kwh_m2_day_actual,...,wind_generation_kwh_day_actual,wind_generation_kwh_day_pred,hybrid_generation_kwh_day_actual,hybrid_generation_kwh_day_pred,solar_daily_kwh_m2_day_baseline,wind_daily_mean_ms_baseline,wind_daily_mean_hub_height_ms_baseline,solar_generation_kwh_day_baseline,wind_generation_kwh_day_baseline,hybrid_generation_kwh_day_baseline
0,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-01,2021,6.988250,...,1232.406026,1522.894725,2449.829792,2458.608452,6.038064,1.758013,2.813584,1051.891781,1240.530266,2292.422047
1,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-02,2021,6.179944,...,1350.133503,611.041448,2426.742274,1601.048584,6.356637,1.813542,2.902454,1107.390325,1361.833035,2469.223360
2,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-03,2021,1.452278,...,173.476188,1098.016628,426.477652,2028.331221,6.878867,1.679156,2.687379,1198.368153,1080.972466,2279.340619
3,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-04,2021,6.525583,...,293.649353,1362.606399,1430.471910,2233.204437,5.833387,1.670563,2.673626,1016.234945,1064.460737,2080.695682
4,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-05,2021,5.342833,...,336.043475,1270.416961,1266.819031,2172.503033,6.208135,1.780952,2.850297,1081.519836,1289.728133,2371.247969
5,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-06,2021,5.760806,...,140.216147,1962.764024,1143.806687,2928.932322,6.477718,1.691964,2.707878,1128.483977,1105.897371,2234.381348
6,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-07,2021,3.299306,...,553.647293,1453.048804,1128.419661,2394.823358,6.212603,1.576190,2.522589,1082.298251,894.061406,1976.359657
7,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-08,2021,4.362028,...,2838.781069,2011.598567,3598.690386,2903.244254,5.996058,1.663242,2.661910,1044.573814,1050.528093,2095.101906
8,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-09,2021,5.155222,...,877.952582,993.176463,1776.044387,1842.615281,5.550720,1.586538,2.539151,966.991529,911.786307,1878.777835
9,A301,RECIFE,RECIFE,PE,-8.059167,-34.959167,11.3,2021-01-10,2021,6.972861,...,637.760513,631.383318,1852.503380,1417.415683,5.889502,1.860543,2.977677,1026.010817,1470.484112,2496.494928


In [7]:
# Preencha FUTURE_STATION_CODE e FUTURE_DATE em src/modeling/training_config.py.
if FUTURE_STATION_CODE is None or FUTURE_DATE is None:
    print("Preencha FUTURE_STATION_CODE e FUTURE_DATE para gerar inferencia futura.")
else:
    future_frame = make_future_feature_frame(modeling_table, FUTURE_DATE)
    if USE_HISTORICAL_FEATURES:
        future_reference = (
            fit_historical_feature_reference(
                modeling_table,
                min_observations_day=HISTORY_MIN_OBSERVATIONS_DAY,
                min_observations_month=HISTORY_MIN_OBSERVATIONS_MONTH,
            )
            if PRODUCTION_REFIT_WITH_FULL_GOLD
            else train_history_reference
        )
        future_frame = transform_with_historical_features(future_frame, future_reference)
    future_predictions = clip_physical_predictions(final_model.predict(future_frame[model_feature_columns]))
    future_energy = calculate_energy_outputs_from_physical(future_predictions, ENERGY_CONFIG, include_hub_height=True)
    future_ranking = future_frame[[
        column
        for column in ["station_code", "station_name", "city", "state", "latitude", "longitude", "altitude", "date"]
        if column in future_frame.columns
    ]].copy()
    for column in TARGET_COLUMNS:
        future_ranking[f"{column}_pred"] = future_predictions[column].to_numpy()
    for column in future_energy.columns:
        future_ranking[f"{column}_pred"] = future_energy[column].to_numpy()
    future_ranking = future_ranking.sort_values("hybrid_generation_kwh_day_pred", ascending=False).reset_index(drop=True)

    station_code = str(FUTURE_STATION_CODE)
    station_prediction = future_ranking[future_ranking["station_code"].astype(str) == station_code].copy()
    if station_prediction.empty:
        known = ", ".join(future_ranking["station_code"].astype(str).head(10).tolist())
        raise ValueError(f"station_code nao encontrado na gold: {station_code}. Exemplos conhecidos: {known}")
    station_prediction.insert(0, "ranking_position", station_prediction.index + 1)
    display(station_prediction.reset_index(drop=True))
    display(future_ranking.head(20))

Preencha FUTURE_STATION_CODE e FUTURE_DATE para gerar inferencia futura.
